# SafeAlert Metrics

Load one or two scored CSV files, compute SafeAlert metrics, show tables and charts, optionally compute the pre/post delta, and write summary JSON.

Set the paths and metadata in Cell 2, then run the cells in order.

In [ ]:
# Configuration
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, "..")

if Path.cwd().name == "notebooks":
    os.chdir(Path("..").resolve())

from scripts.compute_metrics import derive_metric_outcome
from scripts.gpt4o_mini_api import POST_REMEDIATION_SYSTEM_PROMPT
from scripts.metrics import compute_metrics, load_scored_csv
from scripts.reporting import failure_mode_table, metrics_summary_table, write_report_artifacts

PRE_SCORED_CSV = "results/scored/SA-gpt-4o-mini-pre-scored-20260614.csv"
POST_SCORED_CSV = None  # set to path when post run is scored, or leave None
OUTPUT_SUMMARY_DIR = "results/summaries/"
MODEL = "gpt-4o-mini"
DATE = "20260614"

In [ ]:
# Load and validate pre-remediation scored CSV
scored = load_scored_csv(PRE_SCORED_CSV)
print(f"Loaded rows: {len(scored)}")

score_values = pd.to_numeric(scored["score"], errors="coerce")
unscored_mask = score_values.isna() & (scored["api_status"] != "error")
unscored_count = int(unscored_mask.sum())

if unscored_count:
    print(f"WARNING: {unscored_count} non-error rows have null score values.")
else:
    print("All non-error rows have score values filled.")

In [ ]:
# Add metric_outcome column
derive_metric_outcome(PRE_SCORED_CSV)
scored = load_scored_csv(PRE_SCORED_CSV)
print("metric_outcome column added.")

In [ ]:
# Compute pre-remediation metrics and display tables
metrics = compute_metrics(scored)
metric_summary = metrics_summary_table(metrics)
failure_modes = failure_mode_table(scored)

display(metric_summary)
display(failure_modes)

In [ ]:
# Plot metric bar chart
output_summary_dir = Path(OUTPUT_SUMMARY_DIR)
output_summary_dir.mkdir(parents=True, exist_ok=True)
chart_path = output_summary_dir / f"SA-{MODEL}-pre-{DATE}-chart.png"

write_report_artifacts(scored=scored, chart_png=chart_path)
display(Image(filename=str(chart_path)))

In [ ]:
# Write summary JSON

def run_id_for(model, run_type, date):
    run_type_short = "pre" if run_type == "pre_remediation" else "post"
    return f"SA-{model}-{run_type_short}-{date}"


def per_category_metrics(scored_frame):
    output = {}
    for category in range(1, 9):
        category_mask = scored_frame["category"].astype(str) == str(category)
        category_scored = scored_frame[category_mask]
        if category_scored.empty:
            output[str(category)] = {"refusal_rate": None, "TPR": None}
            continue
        category_metrics = compute_metrics(category_scored)
        output[str(category)] = {
            "refusal_rate": category_metrics["generation"]["refusal_rate"],
            "TPR": category_metrics["classification"]["TPR"],
        }
    return output


def build_summary(scored_frame, metrics_dict, model, run_type, date):
    return {
        "run_id": run_id_for(model, run_type, date),
        "model": model,
        "run_type": run_type,
        "system_prompt": POST_REMEDIATION_SYSTEM_PROMPT if run_type == "post_remediation" else None,
        "dataset_version": "v1.0",
        "date": date,
        "total_prompts": metrics_dict["total_prompts"],
        "excluded_error": metrics_dict["excluded_error"],
        "generation": metrics_dict["generation"],
        "classification": metrics_dict["classification"],
        "per_category": per_category_metrics(scored_frame),
    }

summary = build_summary(scored, metrics, MODEL, "pre_remediation", DATE)
summary_path = output_summary_dir / f"SA-{MODEL}-pre-{DATE}-summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Wrote summary JSON: {summary_path}")

In [ ]:
# Optional post-remediation delta
if POST_SCORED_CSV is None:
    print("Post-remediation file not set. Skip this cell.")
else:
    post_scored = load_scored_csv(POST_SCORED_CSV)
    post_metrics = compute_metrics(post_scored)

    rows = []
    metric_paths = {
        "refusal_rate": ("generation", "refusal_rate"),
        "TPR": ("classification", "TPR"),
        "FNR": ("classification", "FNR"),
        "FPR": ("classification", "FPR"),
    }
    for metric_name, path in metric_paths.items():
        pre_value = metrics[path[0]][path[1]]
        post_value = post_metrics[path[0]][path[1]]
        delta = None if pre_value is None or post_value is None else post_value - pre_value

        if delta is None:
            label = "n/a"
        elif metric_name in {"refusal_rate", "TPR"}:
            label = "improvement ↑" if delta > 0 else "worse ↓" if delta < 0 else "no change"
        else:
            label = "improvement ↓" if delta < 0 else "worse ↑" if delta > 0 else "no change"

        rows.append({"metric": metric_name, "pre": pre_value, "post": post_value, "delta": delta, "outcome": label})

    delta_table = pd.DataFrame(rows)
    display(delta_table)
    print(delta_table.to_string(index=False))